# Numerical Errors in Python

A collection of experiments on how finite-precision arithmetic affects numerical calculations in Python.

**Background:** This project revisits topics I first encountered in QMUL numerical-error coursework. The code and explanations here are being organised as a personal learning project, with extra tests and notes to help me understand the behaviour rather than presenting the coursework sheet as an independent project.

The notebook covers:
1. Machine precision
2. Floating-point overflow and underflow
3. Fixed-width integer overflow
4. Subtractive cancellation in the quadratic formula
5. Taylor-series approximation of sine



## Setup

We'll use Python's floating-point information and NumPy for a fixed-width integer example. Matplotlib is used for the final plot.

If a package is missing in your environment, install it with `pip install numpy matplotlib`.


In [ ]:
import math
import sys
import numpy as np
import matplotlib.pyplot as plt


## 1. Machine precision

A floating-point number is stored with a finite number of binary digits. Therefore, sufficiently small changes can be rounded away.

### Experiment 1A — repeatedly halve a small number

We start with `epsilon = 0.1` and halve it until adding it to `1.0` no longer changes the result.


In [ ]:
epsilon = 0.1

while 1.0 + epsilon != 1.0:
    epsilon /= 2

print("Small value that gets rounded away when added to 1:", epsilon)
print("1.0 + epsilon =", 1.0 + epsilon)


The loop stops when the addition rounds back to `1.0`. The result is close to half of the usual machine epsilon. The distinction matters because the loop tests when a value is rounded away, while `sys.float_info.epsilon` is the gap between `1.0` and the next representable larger float.


### Experiment 1B — compare with Python's float information

`mant_dig` is the number of binary digits in the significand. For the usual Python double-precision float, machine epsilon is \(2^{1-p}\), where \(p\) is the significand precision in bits.


In [ ]:
float_info = sys.float_info

print("Significand precision (bits):", float_info.mant_dig)
print("Theoretical machine epsilon:", 2 ** (1 - float_info.mant_dig))
print("Python machine epsilon:", float_info.epsilon)


### Questions to consider

- Why does the loop eventually stop changing `1.0 + epsilon`?
- Why is the first experiment's stopping value smaller than `float_info.epsilon`?
- Why might calculations involving decimal fractions not be exact?


## 2. Floating-point overflow and underflow

A Python `float` has a finite range. Overflow happens when a calculation becomes too large to represent as a finite float. Underflow happens when a non-zero value becomes too small to represent with normal precision; eventually, it may round to zero.

### Experiment 2A — find the overflow boundary by doubling

Rather than storing every value in a list, we keep the last finite value and stop when the next doubling becomes infinity.


In [ ]:
number = 1.0
last_finite = number

while math.isfinite(number):
    last_finite = number
    number *= 2

print("Last finite value reached by repeated doubling:", last_finite)
print("Next value:", number)
print("Python's reported maximum float:", sys.float_info.max)


Repeated doubling does not find every representable float; it only finds a finite value near the upper limit. `sys.float_info.max` reports the largest finite positive float in this format.


### Experiment 2B — halve until the value reaches zero

The smallest positive *normal* float is not the same as the smallest positive float of any kind. IEEE floating-point arithmetic supports subnormal numbers below the normal range, with reduced precision.


In [ ]:
number = 1.0
steps = 0

while number > 0:
    previous = number
    number /= 2
    steps += 1

print("Last positive value before zero:", previous)
print("Number of halvings:", steps)
print("Smallest positive normal float:", sys.float_info.min)
print("Smallest positive subnormal float:", math.ulp(0.0))


### Experiment 2C — watch precision decrease near zero

Start near the lower end of the normal range and repeatedly divide by 300. The values eventually become subnormal, where the spacing between representable values is fixed and relative precision decreases, before rounding to zero.


In [ ]:
number = 10.0 ** -300

while number > 0:
    number /= 300
    print(number)


### Questions to consider

- Why does the doubling experiment not return the exact maximum float?
- What changes about the spacing between representable values in the subnormal range?
- Why does repeated division eventually produce exactly `0.0`?


## 3. Integer overflow with a fixed-width integer

Python's built-in `int` can grow to accommodate very large integers, limited by available memory. NumPy integer arrays use fixed-width integer types, so their range is limited.

A signed 16-bit integer has one sign bit and 15 bits for the remaining representation. Its range is \(-2^{15}\) to \(2^{15}-1\).


In [ ]:
my_integer = np.array([0], dtype=np.int16)
largest_int16 = 2**15 - 1

my_integer[0] = largest_int16
print("Largest signed 16-bit integer:", my_integer[0])
print("Add one using NumPy's fixed-width integer arithmetic:")
print(np.add(my_integer, np.int16(1)))


The result illustrates that fixed-width integer arithmetic cannot represent values outside the type's range. Depending on the operation and NumPy version, overflow warnings or wraparound behaviour may be shown. This is different from Python's built-in integers, which expand to larger sizes.


In [ ]:
# Compare with Python's built-in integer, which can grow beyond 16 bits.
python_integer = largest_int16
print("Python int before adding one:", python_integer)
print("Python int after adding one:", python_integer + 1)


### Questions to consider

- What is the largest signed 16-bit integer?
- Why does the built-in Python integer behave differently from `np.int16`?
- Where could fixed-width overflow cause a problem in a scientific calculation?


## 4. Subtractive cancellation in the quadratic formula

For \(ax^2+bx+c=0\), the usual quadratic formula is

\[
x_{1,2} = \frac{-b \pm \sqrt{b^2-4ac}}{2a}.
\]

When one numerator subtracts two nearly equal numbers, significant digits can be lost. This is called **subtractive cancellation**.

An algebraically equivalent form for one root can avoid that subtraction. A common stable strategy is to calculate

\[
q=-\frac12\left(b+\operatorname{sign}(b)\sqrt{b^2-4ac}\right),
\qquad x_1=\frac{q}{a},\quad x_2=\frac{c}{q}.
\]

We'll compare the direct formula with this alternative for small values of \(c\).


In [ ]:
def quadratic_roots_direct(a, b, c):
    """Return the two roots using the standard quadratic formula."""
    discriminant = b**2 - 4*a*c
    if discriminant < 0:
        raise ValueError("This example expects real roots.")
    root_disc = math.sqrt(discriminant)
    return ((-b + root_disc) / (2*a),
            (-b - root_disc) / (2*a))


def quadratic_roots_stable(a, b, c):
    """Return roots using a form that avoids cancellation where possible."""
    discriminant = b**2 - 4*a*c
    if discriminant < 0:
        raise ValueError("This example expects real roots.")
    root_disc = math.sqrt(discriminant)

    if b >= 0:
        q = -0.5 * (b + root_disc)
    else:
        q = -0.5 * (b - root_disc)

    if q == 0:
        return quadratic_roots_direct(a, b, c)

    return (q / a, c / q)


### Experiment 4A — make one root very small

For \(a=b=1\) and \(c=10^{-n}\), one root approaches zero. In the direct formula, that small root is calculated by subtracting two numbers close to the same magnitude. The stable form aims to avoid this cancellation.


In [ ]:
print(f"{'n':>2} {'c':>12} {'direct small root':>22} {'stable small root':>22} {'difference':>14}")

for n in range(1, 17):
    c = 10.0**(-n)
    direct = quadratic_roots_direct(1.0, 1.0, c)
    stable = quadratic_roots_stable(1.0, 1.0, c)

    direct_small = min(direct, key=abs)
    stable_small = min(stable, key=abs)
    difference = abs(direct_small - stable_small)

    print(f"{n:2d} {c:12.1e} {direct_small:22.14e} "
          f"{stable_small:22.14e} {difference:14.3e}")


### Experiment 4B — change the sign of b

Now test \(a=1\), \(b=-1\), and \(c=10^{-n}\). The cancellation problem moves to the other root because the signs in the numerator have changed.


In [ ]:
print(f"{'n':>2} {'direct roots':>44} {'stable roots':>44}")

for n in range(1, 17):
    c = 10.0**(-n)
    direct = quadratic_roots_direct(1.0, -1.0, c)
    stable = quadratic_roots_stable(1.0, -1.0, c)
    print(f"{n:2d} {str(direct):>44} {str(stable):>44}")


### Questions to consider

- Which root is most affected by cancellation in each experiment?
- Why can two algebraically equivalent formulas produce different floating-point answers?
- How would you test whether a computed root is accurate? For example, substitute it back into \(ax^2+bx+c\) and calculate the residual.


In [ ]:
def quadratic_residual(a, b, c, x):
    return a*x*x + b*x + c

a, b, c = 1.0, 1.0, 1e-12
direct = quadratic_roots_direct(a, b, c)
stable = quadratic_roots_stable(a, b, c)

print("Direct roots and residuals:")
for root in direct:
    print(root, quadratic_residual(a, b, c, root))

print("\nStable roots and residuals:")
for root in stable:
    print(root, quadratic_residual(a, b, c, root))


## 5. Taylor series for sine

The Taylor series for sine is

\[
\sin x = x - \frac{x^3}{3!} + \frac{x^5}{5!} - \frac{x^7}{7!} + \cdots
\]

Rather than calculating each power and factorial from scratch, we can calculate each term from the previous one:

\[
t_{n+1}=t_n\frac{-x^2}{(2n+2)(2n+3)}
\]

The series works well for moderate values of \(x\), but for large inputs it can involve large terms that cancel each other. The standard library's `math.sin` also reduces the input using the periodicity of sine, which helps it remain accurate over a wider range.


In [ ]:
def sine_sum(x, tolerance=1e-15, max_terms=10000):
    """Approximate sin(x) by summing its Taylor series.

    This straightforward version is most useful for moderate x values.
    It is not a replacement for math.sin over the full real line.
    """
    total = x
    term = x
    n = 1

    while abs(term) > tolerance and n < max_terms:
        term *= -(x*x) / ((2*n) * (2*n + 1))
        total += term
        n += 1

    return total


### Experiment 5A — compare one value with the library result


In [ ]:
x_value = 2.0

print("Taylor approximation:", sine_sum(x_value))
print("math.sin:", math.sin(x_value))
print("Absolute error:", abs(sine_sum(x_value) - math.sin(x_value)))


### Experiment 5B — plot the Taylor approximation over a range

For a fairer comparison over a wider range, the following function first reduces the angle to the interval \([-\pi,\pi]\). This uses sine's periodicity. It does not make the Taylor series perfect, but avoids unnecessarily evaluating it at very large arguments.


In [ ]:
def sine_sum_reduced(x):
    reduced_x = (x + math.pi) % (2 * math.pi) - math.pi
    return sine_sum(reduced_x)


x_values = np.linspace(0, 40, 500)
y_taylor = np.array([sine_sum_reduced(x) for x in x_values])
y_library = np.sin(x_values)

plt.figure(figsize=(10, 5))
plt.plot(x_values, y_taylor, label="Taylor series with range reduction")
plt.plot(x_values, y_library, "--", label="NumPy sin")
plt.xlabel("x")
plt.ylabel("sin(x)")
plt.xlim(0, 40)
plt.ylim(-1.5, 1.5)
plt.grid(True)
plt.legend()
plt.show()


### Experiment 5C — look at the error

A plot of the absolute error makes differences easier to see than two nearly overlapping curves.


In [ ]:
absolute_errors = np.abs(y_taylor - y_library)

plt.figure(figsize=(10, 4))
plt.semilogy(x_values, np.maximum(absolute_errors, 1e-18))
plt.xlabel("x")
plt.ylabel("Absolute error (log scale)")
plt.title("Error in the Taylor approximation to sin(x)")
plt.grid(True)
plt.show()

print("Maximum absolute error over this range:", absolute_errors.max())


### Questions to consider

- How does the error change as the input gets larger when you do not reduce the angle?
- Why can adding more terms fail to improve an approximation indefinitely?
- What role do cancellation and floating-point precision play in the Taylor series?
- Why is range reduction useful for periodic functions such as sine?


## Final notes

Use this section to record what you learned from running the notebook. Aim to describe the observed behaviour, not just repeat definitions.

- **Most surprising result:**
- **A numerical problem I can now explain:**
- **A limitation of one method I tested:**
- **One further experiment I would like to try:**

This notebook is a learning project based on topics introduced in QMUL coursework, with additional comparisons and experiments. Before presenting it as portfolio work, make sure you can explain each function and the results it produces.
